This model uses a networked representation of space as the substrate upon which agents move, and trade. (Unlike in the Virus network, in which the nodes themselves were the agents).

In [ ]:
import networkx
import numpy
import matplotlib.pyplot as plt
%matplotlib inline

from BoltzmannWealthModel import BoltzmannWealthModelNetwork

In [ ]:
model = BoltzmannWealthModelNetwork(num_agents=7, num_nodes=10)

In [ ]:
model.run_model(15) # 10 iterations

In [ ]:
model_out = model.datacollector.get_model_vars_dataframe()

In [ ]:
results = model_out.plot()

Go get the Orbis data

In [ ]:
!curl https://raw.githubusercontent.com/sfsheath/gorbit/master/gorbit-edges.csv > orbis.csv

In [ ]:
import pandas as pd

df = pd.read_csv('orbis.csv')
Graphtype = networkx.Graph()
ORBIS = networkx.from_pandas_edgelist(df, edge_attr='km', create_using=Graphtype)

In [ ]:
import random

from mesa import Agent, Model
from mesa.datacollection import DataCollector
import networkx as nx

from mesa.space import NetworkGrid


def compute_gini(model):
    agent_wealths = [agent.wealth for agent in model.agents]
    x = sorted(agent_wealths)
    N = model.num_agents
    B = sum(xi * (N - i) for i, xi in enumerate(x)) / (N * sum(x))
    return 1 + (1 / N) - 2 * B


class BoltzmannWealthModelNetwork2(Model):
    """A model with some number of agents."""

    def __init__(self, num_agents=7, num_nodes=10, **kwargs):
        super().__init__(**kwargs)
        self.num_agents = ORBIS.number_of_nodes()
        self.num_nodes = ORBIS.number_of_nodes()#num_nodes if num_nodes >= self.num_agents else self.num_agents
        self.G = ORBIS #nx.erdos_renyi_graph(n=self.num_nodes, p=0.5)
        self.grid = NetworkGrid(self.G)
        self.datacollector = DataCollector(
            model_reporters={"Gini": compute_gini},
            agent_reporters={"Wealth": lambda _: _.wealth}
        )

        list_of_random_nodes = random.sample(list(self.G.nodes()), self.num_agents)

        # Create agents
        for i in range(self.num_agents):
            a = MoneyAgent(self)
            # Add the agent to a random node
            self.grid.place_agent(a, list_of_random_nodes[i])

        self.running = True
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        # collect data
        self.datacollector.collect(self)

    def run_model(self, n):
        for i in range(n):
            self.step()


class MoneyAgent(Agent):
    """ An agent with fixed initial wealth."""

    def __init__(self, model):
        super().__init__(model)
        self.wealth = 1

    def move(self):
        # Get neighboring node IDs from the graph
        neighbor_nodes = list(self.model.G.neighbors(self.pos))
        # Filter to empty nodes
        possible_steps = [node for node in neighbor_nodes if self.model.grid.is_cell_empty(node)]
        if len(possible_steps) > 0:
            new_position = random.choice(possible_steps)
            self.model.grid.move_agent(self, new_position)

    def give_money(self):
        # Get agents at neighboring nodes
        neighbor_nodes = list(self.model.G.neighbors(self.pos))
        neighbors = self.model.grid.get_cell_list_contents(neighbor_nodes)
        if len(neighbors) > 0:
            other = random.choice(neighbors)
            other.wealth += 1
            self.wealth -= 1

    def step(self):
        self.move()
        if self.wealth > 0:
            self.give_money()

In [ ]:
model2 = BoltzmannWealthModelNetwork2(num_agents=1200) # defaults to 1 agent per node, eg 667 agents, nodes

In [ ]:
model2.run_model(15) # 10 iterations

In [ ]:
model_out2 = model2.datacollector.get_model_vars_dataframe()

In [ ]:
results = model_out2.plot()

In [ ]:
from mesa.batchrunner import batch_run

fixed_params = dict(num_nodes = ORBIS.number_of_nodes())

# Vary density from 0.01 to 1 by increments; np.linspace takes the start, finish, and then
# number of equal spaced samples to generate within that range. 
variable_params = dict(num_agents=numpy.linspace(600,1200,10)[1:]
                       )

In [ ]:
params = {**fixed_params, **variable_params}

results = batch_run(
    BoltzmannWealthModelNetwork2,
    parameters=params,
    rng=[None] * 10,
    max_steps=10,
    data_collection_period=-1,
    display_progress=True,
)

import pandas as pd
df = pd.DataFrame(results)

In [ ]:
df.head()

In [ ]:
plt.scatter(df.num_agents, df.Gini)
plt.grid(True)